<a href="https://colab.research.google.com/github/yesahek/CatsVsDogs-Image-Classifier/blob/main/notebooks/preprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
from pathlib import Path
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Find a sample Image

In [7]:
#  Find a sample image
# Look in processed/train/cat first, then fall back to raw/Cat
root_dir = Path("/content/drive/MyDrive/cats_vs_dogs_classifier/data/")

sample_path = None
search_dirs = [
    Path(f"{root_dir}/processed/train/cat"),
    Path(f"{root_dir}/processed/train/dog"),
    Path(f"{root_dir}/raw/Cat"),
    Path(f"{root_dir}/raw/Dog"),
]
for d in search_dirs:
    if d.exists():
        files = list(d.iterdir())
        if files:
            sample_path = files[0]
            break

In [8]:
print(sample_path)

/content/drive/MyDrive/cats_vs_dogs_classifier/data/processed/train/cat/cat_8630.jpg


In [9]:
if sample_path is None:
    print("No images found. Run script 02 first to split the dataset.")
    sys.exit(1)

In [10]:
print(f"Using sample image: {sample_path}")

Using sample image: /content/drive/MyDrive/cats_vs_dogs_classifier/data/processed/train/cat/cat_8630.jpg


In [11]:
# Load the raw image
import tensorflow as tf
from PIL import Image

raw_img = Image.open(sample_path).convert("RGB")
raw_arr = np.array(raw_img, dtype=np.float32)

print(f"""
RAW IMAGE (straight from disk):
  Size   : {raw_img.size[0]} x {raw_img.size[1]} pixels
  Shape  : {raw_arr.shape}   (height, width, RGB channels)
  Dtype  : {raw_arr.dtype}
  Min px : {raw_arr.min():.0f}
  Max px : {raw_arr.max():.0f}
  Mean px: {raw_arr.mean():.1f}
""")


RAW IMAGE (straight from disk):
  Size   : 500 x 432 pixels
  Shape  : (432, 500, 3)   (height, width, RGB channels)
  Dtype  : float32
  Min px : 0
  Max px : 255
  Mean px: 143.0



In [12]:
# Resize ────────────────────────────────────────────────────────────
IMG_SIZE = (224, 224)

img_tensor  = tf.constant(raw_arr)                     # convert to tensor
img_resized = tf.image.resize(img_tensor, IMG_SIZE)    # resize to 224x224

print(f"""AFTER RESIZE to {IMG_SIZE}:
  Shape  : {img_resized.shape}
  Min px : {img_resized.numpy().min():.0f}
  Max px : {img_resized.numpy().max():.0f}
  (pixel values unchanged — only the dimensions changed)
""")

AFTER RESIZE to (224, 224):
  Shape  : (224, 224, 3)
  Min px : 0
  Max px : 255
  (pixel values unchanged — only the dimensions changed)



In [13]:
# Normalise ─────────────────────────────────────────────────────────
img_normalised = tf.cast(img_resized, tf.float32) / 255.0

print(f"""AFTER NORMALISATION (÷ 255):
  Shape  : {img_normalised.shape}
  Dtype  : {img_normalised.dtype}
  Min px : {img_normalised.numpy().min():.4f}
  Max px : {img_normalised.numpy().max():.4f}
  Mean px: {img_normalised.numpy().mean():.4f}
  (same image, values now between 0.0 and 1.0)
""")

AFTER NORMALISATION (÷ 255):
  Shape  : (224, 224, 3)
  Dtype  : <dtype: 'float32'>
  Min px : 0.0000
  Max px : 1.0000
  Mean px: 0.5604
  (same image, values now between 0.0 and 1.0)



In [15]:
# Show pixel-level example
print("Example pixel transformation (top-left corner, first 5 pixels):")
print(f"  Before: {img_resized.numpy()[0, :5, 0].astype(int).tolist()}")
print(f"  After : {[round(v, 3) for v in img_normalised.numpy()[0, :5, 0].tolist()]}")
print()
print("Preprocessing done!")

Example pixel transformation (top-left corner, first 5 pixels):
  Before: [207, 207, 207, 207, 208]
  After : [0.812, 0.812, 0.812, 0.813, 0.816]

Preprocessing done!
